## Import and Preprocessing


In [ ]:
import os
import json
import yaml
import pandas as pd

# Set base directory and folders
base_dir = '/Users/User/Desktop/inject/data'
ttx_dirs = ['TTX0', 'TTX1', 'TTX2', 'TTX3', 'TTX4', 'TTX4a', 'TTX4b']

# Initialize storage
gt_data = []
elo_data = []

for ttx in ttx_dirs:
    ttx_path = os.path.join(base_dir, ttx)
    objectives_path = os.path.join(ttx_path, 'objectives.yml')
    elo_path = os.path.join(ttx_path, 'exercise_learning_objectives.jsonl')

    if not os.path.exists(objectives_path):
        print(f"Skipping {ttx}: objectives.yml not found")
        continue
    if not os.path.exists(elo_path):
        print(f"Skipping {ttx}: exercise_learning_objectives.jsonl not found")
        continue

    # === Parse objectives.yml ===
    with open(objectives_path, 'r') as f:
        try:
            objectives = yaml.safe_load(f)
        except yaml.YAMLError as e:
            print(f"Failed to parse YAML in {ttx}: {e}")
            continue

        for obj in objectives:
            obj_name = obj.get('name', '')
            obj_tags = obj.get('tags', '')
            for activity in obj.get('activities', []):
                act_name = activity.get('name', '')
                act_tags = activity.get('tags', '')
                milestones = activity.get('milestones', [])
                if not milestones:
                    # Still store activities without milestones
                    gt_data.append({
                        'milestone_id': None,
                        'objective_name': obj_name,
                        'activity_name': act_name,
                        'activity_tags': act_tags,
                        'objective_tags': obj_tags,
                        'source': 'objectives.yml',
                        'ttx': ttx
                    })
                for milestone in milestones:
                    gt_data.append({
                        'milestone_id': milestone,
                        'objective_name': obj_name,
                        'activity_name': act_name,
                        'activity_tags': act_tags,
                        'objective_tags': obj_tags,
                        'source': 'objectives.yml',
                        'ttx': ttx
                    })

    # === Parse exercise_learning_objectives.jsonl ===
    with open(elo_path, 'r') as f:
        for line in f:
            try:
                entry = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Failed to parse JSON in {ttx}: {e}")
                continue

            obj_id = entry.get('objective_id')
            obj_tags = entry.get('tags', '')

            for activity in entry.get('activities', []):
                act_name = activity.get('name', '')
                act_tags = activity.get('tags', '')
                milestones = activity.get('milestone_ids', [])

                if not milestones:
                    # Store activity even if no milestone ID
                    elo_data.append({
                        'milestone_id': None,
                        'objective_id': obj_id,
                        'activity_name': act_name,
                        'activity_tags': act_tags,
                        'objective_tags': obj_tags,
                        'source': 'exercise_learning_objectives.jsonl',
                        'ttx': ttx
                    })
                for milestone in milestones:
                    elo_data.append({
                        'milestone_id': f"I_{milestone}",
                        'objective_id': obj_id,
                        'activity_name': act_name,
                        'activity_tags': act_tags,
                        'objective_tags': obj_tags,
                        'source': 'exercise_learning_objectives.jsonl',
                        'ttx': ttx
                    })

# Convert to DataFrames
df_gt = pd.DataFrame(gt_data)
df_elo = pd.DataFrame(elo_data)

# Merge on milestone ID and ttx
df_merged = pd.merge(
    df_gt, df_elo,
    on=['milestone_id', 'ttx'],
    how='outer',
    suffixes=('_gt', '_elo')
)
df_merged['matched'] = df_merged['source_gt'].notna() & df_merged['source_elo'].notna()

In [ ]:
team_folders = []

for ttx_dir in ['TTX0', 'TTX1', 'TTX2', 'TTX3', 'TTX4', 'TTX4a', 'TTX4b']:
    ttx_path = os.path.join(base_dir, ttx_dir)
    
    # Check if the TTX directory exists
    if os.path.exists(ttx_path):
        # Find all folders starting with 'team-'
        for folder_name in os.listdir(ttx_path):
            if folder_name.startswith('team-') and os.path.isdir(os.path.join(ttx_path, folder_name)):
                team_folders.append((ttx_dir, folder_name))  # Append (TTX directory, team folder)

# Initialize data list to store extracted data
data = []

# Loop through each team folder
for ttx_dir, team_folder in team_folders:
    team_path = os.path.join(base_dir, ttx_dir, team_folder)  # Full path to the team folder
    
    # Check if the milestone file exists for the team
    milestone_file = os.path.join(team_path, 'milestones.jsonl')
    
    # Read the milestone data from JSONL file
    if os.path.exists(milestone_file):
        with open(milestone_file, 'r') as f:
            for line in f:
                milestone = json.loads(line)
                
                # Extract data from the milestone
                teamID = team_folder.split('-')[1]  # Extract team ID from folder name
                milestoneID = milestone['milestone_id']
                achieved = milestone['reached']
                timestamp = milestone['timestamp_reached']
                
                # Append the extracted data
                data.append([teamID, milestoneID, achieved, timestamp])

In [ ]:
df = pd.DataFrame(data, columns=["teamID", "milestoneID", "achieved", "timestamp"])

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(by=['teamID', 'milestoneID'])
df['timestamp_numeric'] = df['timestamp'].astype('int64')  # Convert to nanoseconds
df.loc[~df['achieved'], 'timestamp_numeric'] = pd.NA
df['timestamp_numeric'] = df.groupby('teamID', group_keys=False)['timestamp_numeric'].apply(lambda x: x.interpolate(method='linear'))
df['timestamp_numeric'] = df.groupby('teamID')['timestamp_numeric'].ffill().bfill()
df['timestamp'] = pd.to_datetime(df['timestamp_numeric'].astype('int64'))
df = df.drop(columns=['timestamp_numeric'])

In [ ]:
# Initialize data list to store extracted data
data = []

# Loop through each team folder
for ttx_dir, team_folder in team_folders:
    team_path = os.path.join(base_dir, ttx_dir, team_folder)  # Full path to the team folder
    
    # Check if the emails file exists for the team
    emails_file = os.path.join(team_path, 'emails.jsonl')
    
    # Read the emails data from JSONL file
    if os.path.exists(emails_file):
        with open(emails_file, 'r', encoding='utf-8') as f:
            for line in f:
                email_thread = json.loads(line)

                # Extract data from the email thread
                thread_id = email_thread.get('thread_id')
                subject = email_thread.get('subject')
                timestamp = email_thread.get('timestamp')
                participants = email_thread.get('participants', [])

                # Extract individual emails from the thread
                for email in email_thread.get('emails', []):
                    email_id = email.get('email_id')
                    sender_id = email.get('sender_id')
                    email_timestamp = email.get('timestamp')
                    email_content_raw = email['content'].get('raw', '')
                    email_content_rendered = email['content'].get('rendered', '')
                    
                    # Append extracted data
                    data.append([
                        team_folder, thread_id, subject, timestamp,
                        participants, email_id, sender_id,
                        email_timestamp, email_content_raw, email_content_rendered
                    ])

# Convert extracted data into a DataFrame
df_email = pd.DataFrame(data, columns=[
    "teamID", "threadID", "subject", "thread_timestamp", 
    "participants", "emailID", "senderID", 
    "email_timestamp", "email_content_raw", "email_content_rendered"
])
df_email.to_csv('data/emails-final.csv', index=False)

In [ ]:
data = []

# Loop through each team folder
for ttx_dir, team_folder in team_folders:
    team_path = os.path.join(base_dir, ttx_dir, team_folder)  # Full path to the team folder
    
    # Check if the action logs file exists for the team
    action_logs_file = os.path.join(team_path, 'action_logs.jsonl')
    
    # Read the action logs data from JSONL file
    if os.path.exists(action_logs_file):
        with open(action_logs_file, 'r', encoding='utf-8') as f:
            for line in f:
                action_log = json.loads(line)
                
                details = action_log.get('details', {})

                # Extract data
                action_log_id = action_log.get('action_log_id')
                action_type = action_log.get('type')
                timestamp = action_log.get('timestamp')
                thread_id = details.get('thread_id')
                email_id = details.get('email_id')
                sender_id = details.get('sender_id')
                raw_content = details.get('content', {}).get('raw', '')
                rendered_content = details.get('content', {}).get('rendered', '')

                # Append extracted data
                data.append([
                    team_folder, action_log_id, action_type, timestamp,
                    thread_id, email_id, sender_id, 
                    raw_content, rendered_content
                ])

# Convert extracted data into a DataFrame
df_logs = pd.DataFrame(data, columns=[
    "teamID", "actionLogID", "actionType", "timestamp",
    "threadID", "emailID", "senderID", 
    "content_raw", "content_rendered"
])
df_logs.to_csv('data/logs-final.csv', index=False)

## Create Analysis Sample

In [ ]:
df_email_gold = pd.read_csv('[BLINDED GOOGLE SHEET]')

In [ ]:
df['teamID'] = df['teamID'].astype(str)
df_logs['teamID'] = df_logs['teamID'].str.replace('team-', '').astype(str)
df_email_gold['teamID'] = df_email_gold['teamID'].str.replace('team-', '').astype(str)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df_logs['timestamp'] = pd.to_datetime(df_logs['timestamp'])
df_email_gold['email_timestamp'] = pd.to_datetime(df_email_gold['email_timestamp'])

In [ ]:
def to_utc(ts):
    """
    Convert a scalar Timestamp/NaT or Series to tz-aware UTC.
    - If tz-naive: tz_localize('UTC')
    - If tz-aware: tz_convert('UTC')
    """
    if isinstance(ts, pd.Series):
        s = pd.to_datetime(ts, errors='coerce')
        # localize naive, convert aware
        naive = s.dt.tz is None  # this is None if mixed; handle robustly below
        # Mixed tz-awareness can happen: handle per-element
        def _fix(x):
            if pd.isna(x):
                return x
            if x.tzinfo is None:
                return x.tz_localize('UTC')
            return x.tz_convert('UTC')
        return s.apply(_fix)
    else:
        x = pd.to_datetime(ts, errors='coerce')
        if pd.isna(x):
            return x
        if x.tzinfo is None:
            return x.tz_localize('UTC')
        return x.tz_convert('UTC')

def pick_email_text_col(df):
    for c in ["email_content_raw", "email_content_rendered", "match_text", "email_content", "body", "snippet", "content"]:
        if c in df.columns:
            return c
    return None

df['timestamp'] = to_utc(df['timestamp'])
df_logs['timestamp'] = to_utc(df_logs['timestamp'])
df_email_gold['email_timestamp'] = to_utc(df_email_gold['email_timestamp'])

time_threshold = pd.Timedelta(seconds=10)
previous_milestone_times = {}  # teamID -> last UTC Timestamp or None
email_text_col = pick_email_text_col(df_email_gold)

for idx, row in df.iterrows():
    team_id = row['teamID']
    milestone_time = row['timestamp']  # already tz-aware UTC (or NaT)

    if pd.isna(milestone_time):
        # skip or fill defaults
        df.at[idx, 'chained_logs'] = ""
        df.at[idx, 'chained_emails'] = ""
        df.at[idx, 'n_activities'] = 0
        df.at[idx, 'time_taken'] = 0
        df.at[idx, 'n_emails'] = 0
        df.at[idx, 'n_logs'] = 0
        continue

    prev_time = previous_milestone_times.get(team_id, None)

    # Determine window
    if prev_time is None or (milestone_time - prev_time) > time_threshold:
        # (prev, current] or (-inf, current] if prev is None
        logs_before = df_logs[(df_logs['teamID'] == team_id) &
                              (df_logs['timestamp'] <= milestone_time)]
        emails_before = df_email_gold[(df_email_gold['teamID'] == team_id) &
                                      (df_email_gold['email_timestamp'] <= milestone_time)]
        if prev_time is not None:
            logs_before = logs_before[logs_before['timestamp'] > prev_time]
            emails_before = emails_before[emails_before['email_timestamp'] > prev_time]
    else:
        # “single boundary” — treat as one window ending at milestone_time with no lower bound
        logs_before = df_logs[(df_logs['teamID'] == team_id) &
                              (df_logs['timestamp'] <= milestone_time)]
        emails_before = df_email_gold[(df_email_gold['teamID'] == team_id) &
                                      (df_email_gold['email_timestamp'] <= milestone_time)]

    chained_logs = '<|CHAIN|>'.join(
        logs_before['content_raw'].dropna().astype(str).tolist()
    ) if 'content_raw' in logs_before.columns else ""
    df.at[idx, 'chained_logs'] = chained_logs

    if email_text_col and email_text_col in emails_before.columns:
        chained_emails = '<|CHAIN|>'.join(
            emails_before[email_text_col].dropna().astype(str).tolist()
        )
    else:
        chained_emails = ""
    df.at[idx, 'chained_emails'] = chained_emails

    n_logs = len(logs_before)
    n_emails = len(emails_before)
    df.at[idx, 'n_logs'] = int(n_logs)
    df.at[idx, 'n_emails'] = int(n_emails)
    df.at[idx, 'n_activities'] = int(n_logs + n_emails)

    # logs by actionType
    if 'actionType' in logs_before.columns:
        for act, cnt in logs_before['actionType'].value_counts().items():
            df.at[idx, f"n_{str(act).lower()}"] = int(cnt)
        df.at[idx, 'n_unique_action_types'] = int(logs_before['actionType'].nunique())

    # avg log content length
    if n_logs and 'content_raw' in logs_before.columns:
        df.at[idx, 'avg_log_content_length'] = float(
            logs_before['content_raw'].dropna().astype(str).apply(lambda s: len(s.split())).mean()
        )

    # time taken
    df.at[idx, 'time_taken'] = float((milestone_time - prev_time).total_seconds()) if prev_time is not None else 0.0

    # email features
    if n_emails:
        et = emails_before['email_timestamp'].sort_values()
        diffs = et.diff()
        if diffs.notna().any():
            df.at[idx, 'avg_email_response_time'] = float(diffs.dropna().mean().total_seconds())
            df.at[idx, 'max_email_response_time'] = float(diffs.dropna().max().total_seconds())
        else:
            df.at[idx, 'avg_email_response_time'] = 0.0
            df.at[idx, 'max_email_response_time'] = 0.0

        if email_text_col and email_text_col in emails_before.columns:
            df.at[idx, 'avg_email_content_length'] = float(
                emails_before[email_text_col].dropna().astype(str).apply(lambda s: len(s.split())).mean()
            )

        if 'senderID' in emails_before.columns:
            df.at[idx, 'n_unique_senders'] = int(emails_before['senderID'].nunique())

        # Bloom counts (prefer bloom_final)
        bloom_col = 'bloom_final' if 'bloom_final' in emails_before.columns else ('bloom' if 'bloom' in emails_before.columns else None)
        if bloom_col:
            bcounts = emails_before[bloom_col].dropna().value_counts()
            for label, cnt in bcounts.items():
                df.at[idx, f"bloom_count_{str(label)}"] = int(cnt)
            df.at[idx, "bloom_count_total"] = int(bcounts.sum())
        else:
            df.at[idx, "bloom_count_total"] = 0
    else:
        df.at[idx, 'avg_email_response_time'] = 0.0
        df.at[idx, 'max_email_response_time'] = 0.0
        df.at[idx, 'avg_email_content_length'] = 0.0
        df.at[idx, 'n_unique_senders'] = 0
        df.at[idx, "bloom_count_total"] = 0

    # update previous
    previous_milestone_times[team_id] = milestone_time


In [ ]:
df = df.fillna(0)
df.to_csv('data/modeling-data-final.csv', index=False)

## LASSO Modeling (RQ2)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sentence_transformers import SentenceTransformer
from collections import Counter

df = df.drop(columns=['milestoneID', 'timestamp'], errors='ignore')

df['achieved'] = df['achieved'].apply(lambda x: 1 if x else 0)

# Define target and groups
y = df['achieved']
groups = df['teamID']

# Train/test split (stratify by y, preserve group labels)
X = df.drop(columns=['achieved', 'teamID'], errors='ignore')
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.2, stratify=y, random_state=42
)

# Text columns
TEXT_COLS = []
if 'chained_logs' in X_train.columns:
    TEXT_COLS.append('chained_logs')
if 'chained_emails' in X_train.columns:
    TEXT_COLS.append('chained_emails')

# Bloom feature list (user said you already have this object)
# Ensure it's filtered to those present and numeric
bloom_features = [c for c in X_train.columns if ('bloom' in c)] if 'bloom_features' not in globals() else [
    c for c in bloom_features if c in X_train.columns
]

# Numeric columns present
numeric_all = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Bloom numeric subset
bloom_numeric = [c for c in bloom_features if c in numeric_all]

# "Log" numeric = numeric features that are NOT bloom (and not group id)
log_numeric = [c for c in numeric_all if c not in set(bloom_numeric) and c != 'teamID']
bloom_numeric = [c for c in bloom_numeric if c != 'teamID']

# -----------------------------
# Sentence embeddings (text)
# -----------------------------
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_series(s):
    # s must be a list of strings; replace NaNs with empty string
    return embedding_model.encode([x if isinstance(x, str) else "" for x in s])

# Build text embedding matrices
emb_train_parts = []
emb_test_parts = []
for col in TEXT_COLS:
    emb_train_parts.append(embed_series(X_train[col].tolist()))
    emb_test_parts.append(embed_series(X_test[col].tolist()))

if len(emb_train_parts) > 0:
    X_train_text = np.hstack(emb_train_parts) if len(emb_train_parts) > 1 else emb_train_parts[0]
    X_test_text  = np.hstack(emb_test_parts)  if len(emb_test_parts)  > 1 else emb_test_parts[0]
else:
    # No text columns found -> empty arrays
    X_train_text = np.zeros((X_train.shape[0], 0))
    X_test_text  = np.zeros((X_test.shape[0], 0))

# -----------------------------
# Build numeric blocks
# -----------------------------
def get_block(train_df, test_df, cols):
    if len(cols) == 0:
        return np.zeros((train_df.shape[0], 0)), np.zeros((test_df.shape[0], 0))
    return train_df[cols].to_numpy(), test_df[cols].to_numpy()

# Remove teamID from features (group only)
X_train = X_train.drop(columns=['teamID'], errors='ignore')
X_test  = X_test.drop(columns=['teamID'], errors='ignore')

X_train_bloom, X_test_bloom = get_block(X_train, X_test, bloom_numeric)
X_train_log,   X_test_log   = get_block(X_train, X_test, log_numeric)

# -----------------------------
# Define feature set assembler
# -----------------------------
FEATURE_SETS = {
    "Text only":        lambda: (X_train_text, X_test_text),
    "Log only":         lambda: (X_train_log, X_test_log),
    "Bloom only":       lambda: (X_train_bloom, X_test_bloom),
    "Text + Log":       lambda: (np.hstack([X_train_text, X_train_log]),  np.hstack([X_test_text,  X_test_log])),
    "Text + Bloom":     lambda: (np.hstack([X_train_text, X_train_bloom]),np.hstack([X_test_text, X_test_bloom])),
    "Log + Bloom":      lambda: (np.hstack([X_train_log,  X_train_bloom]),np.hstack([X_test_log,  X_test_bloom])),
    "All (Text+Log+Bloom)": lambda: (np.hstack([X_train_text, X_train_log, X_train_bloom]),
                                     np.hstack([X_test_text,  X_test_log,  X_test_bloom])),
}

# -----------------------------
# CI utilities
# -----------------------------
def ci_from_scores(scores, alpha=0.05):
    """95% CI from sample of scores (e.g., CV folds) using t-interval."""
    scores = np.array(scores, dtype=float)
    m = np.mean(scores)
    s = np.std(scores, ddof=1) if len(scores) > 1 else 0.0
    from scipy.stats import t
    n = len(scores)
    if n > 1 and s > 0:
        tcrit = t.ppf(1 - alpha/2, df=n-1)
        half = tcrit * s / np.sqrt(n)
        return m, (m - half, m + half)
    else:
        return m, (m, m)

def stratified_bootstrap_auc(y_true, y_pred, n_boot=2000, random_state=42, alpha=0.05):
    """
    Stratified bootstrap CI for AUC on a test set.
    Keeps class proportions similar in each resample.
    """
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Indices by class
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    n_pos, n_neg = len(idx_pos), len(idx_neg)

    aucs = []
    for _ in range(n_boot):
        # Sample with replacement within each class
        samp_pos = rng.choice(idx_pos, size=n_pos, replace=True)
        samp_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        samp_idx = np.concatenate([samp_pos, samp_neg])
        aucs.append(roc_auc_score(y_true[samp_idx], y_pred[samp_idx]))
    lower = np.percentile(aucs, 100*alpha/2)
    upper = np.percentile(aucs, 100*(1 - alpha/2))
    return float(np.mean(aucs)), (float(lower), float(upper))

# -----------------------------
# Trainer / evaluator
# -----------------------------
def evaluate_feature_set(name, Xtr, Xte, ytr, yte, groups_tr):
    # Pipeline: scale -> L1-logistic (LASSO)
    pipe = Pipeline([
        ('scaler', StandardScaler(with_mean=True, with_std=True)),
        ('clf', LogisticRegression(penalty='l1', solver='liblinear', random_state=42, max_iter=1000))
    ])

    gkf = GroupKFold(n_splits=5)
    cv_scores = cross_val_score(pipe, Xtr, ytr, cv=gkf, scoring='roc_auc', groups=groups_tr, n_jobs=None)
    cv_mean, cv_ci = ci_from_scores(cv_scores)

    # Fit on full training, evaluate on test
    pipe.fit(Xtr, ytr)
    y_prob = pipe.predict_proba(Xte)[:, 1]
    test_auc = roc_auc_score(yte, y_prob)
    boot_mean, boot_ci = stratified_bootstrap_auc(yte, y_prob, n_boot=2000, random_state=123)

    print(f"=== {name} ===")
    print(f"CV AUC (mean ± 95% CI): {cv_mean:.3f} [{cv_ci[0]:.3f}, {cv_ci[1]:.3f}] | folds: {np.round(cv_scores, 3)}")
    print(f"Test AUC: {test_auc:.3f} | Bootstrap 95% CI: [{boot_ci[0]:.3f}, {boot_ci[1]:.3f}] (boot mean {boot_mean:.3f})")
    print("-" * 70)
    return {
        "name": name,
        "cv_mean": cv_mean,
        "cv_ci": cv_ci,
        "cv_folds": cv_scores,
        "test_auc": test_auc,
        "test_boot_ci": boot_ci,
        "test_boot_mean": boot_mean
    }

# -----------------------------
# Run all comparisons
# -----------------------------
results = []
for set_name, builder in FEATURE_SETS.items():
    Xtr, Xte = builder()
    # Guard: skip empty feature sets
    if Xtr.shape[1] == 0:
        print(f"Skipping '{set_name}' (no features).")
        continue
    res = evaluate_feature_set(set_name, Xtr, Xte, y_train, y_test, groups_train)
    results.append(res)

# Optional: summarize best by test AUC
if results:
    best = max(results, key=lambda r: r['test_auc'])
    print(f"\nBest (by Test AUC): {best['name']} — AUC {best['test_auc']:.3f} "
          f"[{best['test_boot_ci'][0]:.3f}, {best['test_boot_ci'][1]:.3f}]")
